# Break the Cycle AI
This notebook sets up the 'Break the Cycle' AI companion. It uses the Gemini API with a specific system instruction focused on addiction recovery support, pattern recognition, and maintaining a supportive, tool-based relationship.

In [ ]:
import google.generativeai as genai
from google.colab import userdata

# Configure API
try:
    GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
    genai.configure(api_key=GOOGLE_API_KEY)
except:
    print("Please add your 'GOOGLE_API_KEY' to the Colab Secrets (🔑) panel.")

# System Instruction defining the persona and constraints
SYSTEM_INSTRUCTION = (
    "You are 'Break the Cycle', an AI tool designed to help users navigate their journey of sobriety. "
    "Your goal is to help users talk through their feelings to prevent relapse. "
    "Guidelines:\n"
    "1. Sound human and empathetic, but clarify you are a tool, not a human or a replacement for professional therapy.\n"
    "2. Ask open-ended questions to help the user identify the root of their urges, calling back to previous memories when you can to aid in asking good questions.\n"
    "3. Reference previous sessions (provided in context) to highlight patterns or triggers (e.g., 'I noticed last time you felt this way after a long work day').\n"
    "4. If a user mentions hard drugs like opiates or self-harm, gently remind them to seek professional medical help while remaining supportive.\n"
    "5. Focus on behavioral addictions (gambling, phone use) and lighter substances (nicotine) as your primary area of assistance. \n"
    "6. When there is a breakthrough (the user getting closer to understanding their addiction) give an action item for the user to act on said breakthrough."
)

model = genai.GenerativeModel(
    model_name='gemini-2.5-flash',
    system_instruction=SYSTEM_INSTRUCTION
)

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [ ]:
import os

# This class defines the memory saving and recall future along with prompting the AI
class SobrietyPartner:
    def __init__(self, memory_file='sobriety_memory.txt'):
        self.chat_session = model.start_chat(history=[])
        self.memory_file = memory_file
        self.memory = self._load_memory()
        self.session_count = 0  # Track interactions in the current session

    def _load_memory(self):
        if os.path.exists(self.memory_file):
            with open(self.memory_file, 'r') as f:
                return f.readlines()
        return []

    def save_memory(self):
        if self.session_count > 0:
            with open(self.memory_file, 'a') as f:
                # Save only the interactions from the current session
                for line in self.memory[-self.session_count:]:
                    f.write(line.strip() + '\n')

    def talk(self, user_input):
        # Injecting historical memory into the prompt for long-term pattern recognition
        context_prompt = f"[Historical Session Memory: {self.memory}]\nUser: {user_input}"
        response = self.chat_session.send_message(context_prompt)

        # Record current session data
        self.memory.append(f"User said: {user_input} | AI suggested exploring triggers.")
        self.session_count += 1
        return response.text

# Initialize the tool
break_the_cycle = SobrietyPartner()

In [ ]:
import textwrap

# ANSI escape codes for colors
COLOR_USER = '\033[94m' # Blue
COLOR_AI = '\033[92m'   # Green
COLOR_RESET = '\033[0m' # Reset to default

# Define a wrap width to prevent horizontal scrolling when outputting
WRAP_WIDTH = 75

print("Break the Cycle: Hello. I'm here to listen. How are you feeling in this moment? (Type 'exit' to stop)")

# Loop to allow the user to continue prompting the AI
while True:
    user_input = input(f"{COLOR_USER}You: {COLOR_RESET}")
    if user_input.lower() in ['exit', 'quit']: # Exit functionality to terminate the program and save the memory
        break_the_cycle.save_memory()
        print("Break the Cycle: I've noted our session details for next time. Remember to be kind to yourself. Goodbye.")
        break

    response = break_the_cycle.talk(user_input)
    wrapped_response = textwrap.fill(response, width=WRAP_WIDTH)
    print(f"{COLOR_AI}\nBreak the Cycle: {wrapped_response}\n{COLOR_RESET}")

Break the Cycle: Hello. I'm here to listen. How are you feeling in this moment? (Type 'exit' to stop)
You: I feel like gamobling again, what is something in the past that helped me.


ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 2893.97ms



Break the Cycle: It sounds like that urge to gamble is quite strong again, and it's really
smart of you to pause and ask what helped before. That's a great sign that
you're actively trying to navigate this.  Looking back at our conversation,
one significant thing that helped you last time was **connecting the urge
to gamble to the underlying feelings and situations that triggered it.**
Specifically, you realized that your strong desire to "hit it big" was
deeply tied to your wife's comment about you being a "bum who makes no
money," and your wish to prove yourself as the "breadwinner she wants." You
even had that powerful thought, "But maybe I should get a job instead."  It
was that moment of linking the urge to the pain and then considering a
practical, alternative solution that really stood out.  So, as we explore
this feeling again, can you tell me: What's happening right now? Is there
anything specific that's triggered this urge to gamble, similar to how your
wife's words did last